# Build Modeling Tables with Cutoff-Safe Features

### Setup

The config controls the airport, tracked airlines, date range, horizons, delay bins, lookback windows, and the assumption that tail number is known at prediction time.

In [1]:
from pathlib import Path
import json
from types import SimpleNamespace
import holidays
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CONFIG_PATH = ROOT / "config" / "project_config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

PATHS = SimpleNamespace(
    processed=ROOT / "data/processed",
    modeling=ROOT / "data/modeling",
)
AIRPORT = CONFIG["airport"]
NOAA_STATION_ID = CONFIG["noaa_station_id"]
TRACKED_AIRLINES = tuple(CONFIG["airlines"])
HORIZONS = tuple(CONFIG["horizons"])
FEATURES = CONFIG["features"]
DELAY_THRESHOLDS = tuple(int(value) for value in CONFIG["bin_edges_minutes"])
BIN_LABELS = CONFIG["bin_labels"]
HISTORICAL_CAUSE_LOOKBACK_DAYS = int(FEATURES["historical_cause_lookback_days"])
WEATHER_PRIOR_WINDOW_HOURS = int(FEATURES["weather_prior_window_hours"])
WEATHER_PRIOR_WINDOW_SUFFIX = f"{WEATHER_PRIOR_WINDOW_HOURS}h"

HISTORICAL_RATE_PRIOR_STRENGTH = 20
RELIABILITY_LEVELS = {
    "carrier": ("reporting_airline",),
    "destination": ("dest",),
    "carrier_destination": ("reporting_airline", "dest"),
}
CAUSE_COLUMNS = {
    "carrier": "carrier_delay_minutes",
    "nas": "nas_delay_minutes",
    "late_aircraft": "late_aircraft_delay_minutes",
    "weather": "weather_delay_minutes",
}
IDENTIFIER_COLUMNS = [
    "flight_id", "flight_date", "scheduled_dep_utc", "prediction_cutoff_utc",
    "reporting_airline", "dest", "prediction_horizon_hours",
]
TARGET_COLUMNS = [
    "dep_delay_minutes", "cancelled", "target_cancelled", "target_delay_ge_15",
    "target_delay_bin", "target_delay_bin_label",
]
EDA_ONLY_COLUMNS = [
    "eda_only_carrier_delay_minutes", "eda_only_weather_delay_minutes",
    "eda_only_nas_delay_minutes", "eda_only_security_delay_minutes",
    "eda_only_late_aircraft_delay_minutes",
]

### Load processed datasets

In [2]:
flights = pd.read_parquet(PATHS.processed / "flights_sfo_touch.parquet")
weather = pd.read_parquet(PATHS.processed / "weather_hourly.parquet")
aircraft = pd.read_parquet(PATHS.processed / "faa_aircraft.parquet")

### Review the SFO departures

The model predicts departures from SFO for the tracked airlines. This quick audit checks volume, cancellations, and fields that will matter later for feature engineering.

In [3]:
sfo_departures = flights.loc[flights["origin"].eq(AIRPORT) & flights["reporting_airline"].isin(TRACKED_AIRLINES)].copy()

bts_columns = [
    "scheduled_dep_utc", "actual_dep_utc", "actual_arr_utc", "tail_number",
    "dep_delay_minutes", "arr_delay_minutes", "cancelled", "diverted",
    "carrier_delay_minutes", "nas_delay_minutes", "late_aircraft_delay_minutes",
]
bts_missing = (100 * sfo_departures[bts_columns].isna().mean()).sort_values(ascending=False)
airline_counts = sfo_departures["reporting_airline"].value_counts().reindex(TRACKED_AIRLINES)

display(pd.DataFrame(airline_counts.rename("SFO departures")))
display(pd.DataFrame(bts_missing.rename("missing_percent").round(2)))

,SFO departures
reporting_airline,
AS,51521
AA,46294
DL,55521
F9,15408
HA,3267
B6,20588
WN,36727
UA,225422


,missing_percent
carrier_delay_minutes,80.27
late_aircraft_delay_minutes,80.27
nas_delay_minutes,80.27
actual_arr_utc,1.42
arr_delay_minutes,1.42
actual_dep_utc,1.18
dep_delay_minutes,1.15
tail_number,0.56
scheduled_dep_utc,0.00
cancelled,0.00


### Cutoff-safe window calculations

The following helpers count only events that were available before each flight's prediction cutoff. They are reused for historical reliability and recent airport operations.


In [4]:
def calculate_window_totals(
    events,
    flights_to_score,
    group_columns,
    event_time,
    metric_columns,
    lookback,
    include_cutoff=False,
):
    """Total prior-event metrics within one cutoff-safe lookback window."""
    def normalize_group_key(value):
        if len(group_columns) == 1 and isinstance(value, tuple):
            return value[0]
        return value

    totals = {
        metric: np.zeros(len(flights_to_score), dtype=float)
        for metric in metric_columns
    }
    if events.empty or flights_to_score.empty:
        return totals

    if group_columns:
        event_groups = {
            normalize_group_key(key): group.index.to_numpy()
            for key, group in events.groupby(
                list(group_columns), observed=True, sort=False
            )
        }
        score_groups = flights_to_score.groupby(
            list(group_columns), observed=True, sort=False
        )
    else:
        event_groups = {"all_flights": events.index.to_numpy()}
        score_groups = [("all_flights", flights_to_score)]

    for raw_key, score_group in score_groups:
        event_index = event_groups.get(normalize_group_key(raw_key))
        if event_index is None:
            continue

        group_events = events.loc[event_index].sort_values(event_time, kind="stable")
        event_times = pd.to_datetime(
            group_events[event_time], utc=True
        ).to_numpy(dtype="datetime64[ns]")
        cutoffs = pd.to_datetime(
            score_group["prediction_cutoff_utc"], utc=True
        ).to_numpy(dtype="datetime64[ns]")

        left = np.searchsorted(
            event_times,
            cutoffs - lookback.to_timedelta64(),
            side="left",
        )
        right = np.searchsorted(
            event_times,
            cutoffs,
            side="right" if include_cutoff else "left",
        )
        row_positions = flights_to_score.index.get_indexer(score_group.index)

        for metric in metric_columns:
            values = pd.to_numeric(
                group_events[metric], errors="coerce"
            ).fillna(0).to_numpy()
            cumulative = np.concatenate([[0.0], np.cumsum(values, dtype=float)])
            totals[metric][row_positions] = cumulative[right] - cumulative[left]

    return totals


In [5]:
def prepare_historical_flights(history):
    """Set when each historical flight's result became available."""
    historical_flights = history.copy()

    actual_arrival = pd.to_datetime(historical_flights["actual_arr_utc"], utc=True)
    actual_departure = pd.to_datetime(historical_flights["actual_dep_utc"], utc=True)
    scheduled_departure = pd.to_datetime(historical_flights["scheduled_dep_utc"], utc=True)
    historical_flights["_available_utc"] = actual_arrival.fillna(actual_departure).fillna(scheduled_departure)

    delay = pd.to_numeric(historical_flights["dep_delay_minutes"], errors="coerce")
    cancelled = pd.to_numeric(historical_flights["cancelled"], errors="coerce").fillna(0).eq(1)
    operated = ~cancelled & delay.notna()

    historical_flights["_total"] = 1.0
    historical_flights["_operated"] = operated.astype(float)
    historical_flights["_cancelled"] = cancelled.astype(float)
    for threshold in DELAY_THRESHOLDS:
        historical_flights[f"_delay_ge_{threshold}"] = (operated & delay.ge(threshold)).astype(float)

    return historical_flights.dropna(subset=["_available_utc"])


def calculate_smoothed_rate(successes, observations, airport_successes, airport_observations, fallback):
    """Stabilize rates for routes with limited historical data."""
    airport_rate = np.divide(
        airport_successes,
        airport_observations,
        out=np.full(len(airport_observations), fallback, dtype=float),
        where=airport_observations > 0,
    )
    strength = HISTORICAL_RATE_PRIOR_STRENGTH
    return (successes + strength * airport_rate) / (observations + strength)

### Historical reliability

For the configured lookback windows, the model receives counts and delay-threshold rates for carrier, destination, and carrier-destination groups.

Why this matters? A route like SFO-EWR can carry different baseline risk than a shorter west-coast route before any same-day disruption is observed.


In [6]:
def add_historical_features(departures, history, lookback_days):
    """Add trailing carrier, destination, and route performance."""
    model_data = departures.copy().reset_index(drop=True)
    historical_flights = prepare_historical_flights(history)
    delay_metrics = [f"_delay_ge_{threshold}" for threshold in DELAY_THRESHOLDS]
    metrics = ["_total", "_operated", "_cancelled", *delay_metrics]
    fallback_rates = {15: 0.20, 60: 0.07, 120: 0.03, 240: 0.01}
    features_created = []

    for days in lookback_days:
        lookback = pd.Timedelta(days=int(days))
        airport_totals = calculate_window_totals(
            historical_flights,
            model_data,
            (),
            "_available_utc",
            metrics,
            lookback,
            include_cutoff=True,
        )

        for level, group_columns in RELIABILITY_LEVELS.items():
            group_totals = calculate_window_totals(
                historical_flights,
                model_data,
                group_columns,
                "_available_utc",
                metrics,
                lookback,
                include_cutoff=True,
            )
            prefix = f"hist_{level}_{int(days)}d"

            new_columns = {
                f"{prefix}_total_count": group_totals["_total"],
                f"{prefix}_operated_count": group_totals["_operated"],
                f"{prefix}_cancel_rate": calculate_smoothed_rate(
                    group_totals["_cancelled"],
                    group_totals["_total"],
                    airport_totals["_cancelled"],
                    airport_totals["_total"],
                    fallback=0.015,
                ),
            }
            for threshold in DELAY_THRESHOLDS:
                new_columns[f"{prefix}_delay_ge_{threshold}_rate"] = calculate_smoothed_rate(
                    group_totals[f"_delay_ge_{threshold}"],
                    group_totals["_operated"],
                    airport_totals[f"_delay_ge_{threshold}"],
                    airport_totals["_operated"],
                    fallback=fallback_rates.get(threshold, 0.01),
                )

            for name, values in new_columns.items():
                model_data[name] = values.astype("float32")
                features_created.append(name)

    return model_data, features_created


### Historical delay causes

Current-flight delay causes would leak the answer. Past carrier, NAS, late-aircraft, and weather cause prevalence is allowed because it describes prior disruption patterns only.

In [7]:
def add_cause_history(departures, history):
    """Add past delay-cause rates without using the current flight's cause."""
    model_data = departures.copy().reset_index(drop=True)
    historical_flights = prepare_historical_flights(history)

    reported_causes = historical_flights[list(CAUSE_COLUMNS.values())].apply(pd.to_numeric, errors="coerce")
    historical_flights["_cause_report"] = reported_causes.notna().any(axis=1).astype(float)
    for cause, column in CAUSE_COLUMNS.items():
        historical_flights[f"_cause_{cause}"] = (
            pd.to_numeric(historical_flights[column], errors="coerce").gt(0)
            & historical_flights["_cause_report"].eq(1)
        ).astype(float)

    metrics = ["_cause_report", *[f"_cause_{cause}" for cause in CAUSE_COLUMNS]]
    lookback_days = HISTORICAL_CAUSE_LOOKBACK_DAYS
    lookback = pd.Timedelta(days=lookback_days)
    airport_totals = calculate_window_totals(
        historical_flights,
        model_data,
        (),
        "_available_utc",
        metrics,
        lookback,
        include_cutoff=True,
    )

    features_created = []
    for level, group_columns in RELIABILITY_LEVELS.items():
        group_totals = calculate_window_totals(
            historical_flights,
            model_data,
            group_columns,
            "_available_utc",
            metrics,
            lookback,
            include_cutoff=True,
        )
        prefix = f"hist_cause_{level}_{lookback_days}d"
        count_name = f"{prefix}_report_count"
        model_data[count_name] = group_totals["_cause_report"].astype("float32")
        features_created.append(count_name)

        for cause in CAUSE_COLUMNS:
            name = f"{prefix}_{cause}_positive_rate"
            model_data[name] = calculate_smoothed_rate(
                group_totals[f"_cause_{cause}"],
                group_totals["_cause_report"],
                airport_totals[f"_cause_{cause}"],
                airport_totals["_cause_report"],
                fallback=0.0,
            ).astype("float32")
            features_created.append(name)

    return model_data, features_created


### Schedule and calendar

These are known before the day of travel: departure hour, arrival hour, month, weekend flag, holiday proximity, distance, planned elapsed time, carrier, destination, and scheduled SFO volume.

In [8]:
def add_schedule_features(departures, all_flights):
    """Add schedule, calendar, and planned-volume information."""
    model_data = departures.copy()
    dep_time = pd.to_numeric(model_data["crs_dep_time"], errors="coerce")
    arr_time = pd.to_numeric(model_data["crs_arr_time"], errors="coerce")
    model_data["scheduled_dep_hour"] = (dep_time // 100) % 24
    model_data["scheduled_arr_hour"] = (arr_time // 100) % 24
    model_data["month"] = pd.to_datetime(model_data["flight_date"]).dt.month
    model_data["is_weekend"] = model_data["day_of_week"].isin([6, 7]).astype("int8")

    model_data["hour_sin"] = np.sin(2 * np.pi * model_data["scheduled_dep_hour"] / 24)
    model_data["hour_cos"] = np.cos(2 * np.pi * model_data["scheduled_dep_hour"] / 24)
    model_data["month_sin"] = np.sin(2 * np.pi * model_data["month"] / 12)
    model_data["month_cos"] = np.cos(2 * np.pi * model_data["month"] / 12)

    dates = pd.to_datetime(model_data["flight_date"]).dt.normalize()
    years = range(int(dates.dt.year.min()) - 1, int(dates.dt.year.max()) + 2)
    holiday_dates = np.array(
        sorted(
            pd.Timestamp(day).to_datetime64()
            for day in holidays.US(years=years)
        ),
        dtype="datetime64[D]",
    )
    date_values = dates.values.astype("datetime64[D]")
    insertion = np.searchsorted(holiday_dates, date_values)
    next_holiday = np.clip(insertion, 0, len(holiday_dates) - 1)
    previous_holiday = np.clip(insertion - 1, 0, len(holiday_dates) - 1)
    model_data["days_until_holiday"] = (holiday_dates[next_holiday] - date_values).astype(int)
    model_data["days_since_holiday"] = (date_values - holiday_dates[previous_holiday]).astype(int)

    sfo_schedule = all_flights.loc[
        all_flights["origin"].eq(model_data["origin"].iloc[0])
    ].copy()
    sfo_schedule["_scheduled_hour"] = sfo_schedule["scheduled_dep_utc"].dt.floor("h")
    airport_counts = sfo_schedule.groupby("_scheduled_hour", observed=True).size()
    carrier_counts = sfo_schedule.groupby(["reporting_airline", "_scheduled_hour"], observed=True).size()

    model_data["_scheduled_hour"] = model_data["scheduled_dep_utc"].dt.floor("h")
    model_data["scheduled_sfo_departures_same_hour"] = model_data["_scheduled_hour"].map(airport_counts).astype("float32")
    carrier_keys = pd.MultiIndex.from_frame(
        model_data[["reporting_airline", "_scheduled_hour"]]
    )
    model_data["scheduled_carrier_departures_same_hour"] = carrier_counts.reindex(carrier_keys).to_numpy(dtype=float)
    model_data = model_data.drop(columns="_scheduled_hour")

    features_created = [
        "day_of_week", "scheduled_dep_hour", "scheduled_arr_hour", "month",
        "is_weekend", "hour_sin", "hour_cos", "month_sin", "month_cos",
        "days_until_holiday", "days_since_holiday",
        "scheduled_sfo_departures_same_hour",
        "scheduled_carrier_departures_same_hour", "distance_miles",
        "scheduled_elapsed_minutes", "reporting_airline", "dest",
    ]
    return model_data, features_created


### Same-tail lineage

The tail identifies the prior leg, but it does not make future outcomes available. Prior actual delays are populated only when the corresponding prior event occurred by the prediction cutoff.

In [9]:
def add_lineage_features(departures, all_flights):
    """Add information from the previous flight assigned to the same tail."""
    tail_flights = all_flights.dropna(subset=["tail_number", "scheduled_dep_utc"]).copy()
    tail_flights = tail_flights.sort_values(["tail_number", "scheduled_dep_utc", "flight_id"])
    tail_groups = tail_flights.groupby("tail_number", sort=False)

    previous_columns = {
        "flight_id": "previous_flight_id",
        "origin": "previous_origin",
        "dest": "previous_dest",
        "scheduled_dep_utc": "previous_scheduled_dep_utc",
        "scheduled_arr_utc": "previous_scheduled_arr_utc",
        "actual_dep_utc": "previous_actual_dep_utc",
        "actual_arr_utc": "previous_actual_arr_utc",
        "dep_delay_minutes": "previous_dep_delay_raw",
        "arr_delay_minutes": "previous_arr_delay_raw",
        "cancelled": "previous_cancelled",
        "diverted": "previous_diverted",
    }
    for source, new_name in previous_columns.items():
        tail_flights[new_name] = tail_groups[source].shift(1)

    tail_flights["tail_sequence_today"] = tail_flights.groupby(
        ["tail_number", tail_flights["flight_date"].dt.date], sort=False
    ).cumcount().add(1)

    model_data = departures.merge(
        tail_flights[
            ["flight_id", "tail_sequence_today", *previous_columns.values()]
        ],
        on="flight_id",
        how="left",
        validate="one_to_one",
    )

    previous_leg_matches = (
        model_data["previous_dest"].eq(model_data["origin"])
        & model_data["previous_flight_id"].notna()
    )
    arrived_by_cutoff = (
        previous_leg_matches
        & model_data["previous_cancelled"].eq(0)
        & model_data["previous_diverted"].eq(0)
        & model_data["previous_actual_arr_utc"].notna()
        & model_data["previous_actual_arr_utc"].le(
            model_data["prediction_cutoff_utc"]
        )
    )
    departed_by_cutoff = (
        previous_leg_matches
        & model_data["previous_actual_dep_utc"].notna()
        & model_data["previous_actual_dep_utc"].le(
            model_data["prediction_cutoff_utc"]
        )
    )

    model_data["has_previous_tail_flight"] = model_data["previous_flight_id"].notna().astype("int8")
    model_data["previous_flight_matches_origin"] = previous_leg_matches.astype("int8")
    model_data["previous_arrived_by_cutoff"] = arrived_by_cutoff.astype("int8")
    model_data["previous_departed_by_cutoff"] = departed_by_cutoff.astype("int8")

    # Realized delays are exposed only after the corresponding event occurred.
    model_data["previous_arrival_delay_known"] = model_data["previous_arr_delay_raw"].where(arrived_by_cutoff)
    model_data["previous_departure_delay_known"] = model_data["previous_dep_delay_raw"].where(departed_by_cutoff)
    model_data["scheduled_turnaround_minutes"] = (
        model_data["scheduled_dep_utc"]
        - model_data["previous_scheduled_arr_utc"]
    ).dt.total_seconds().div(60).where(previous_leg_matches)
    model_data["minutes_until_previous_scheduled_arrival"] = (
        model_data["previous_scheduled_arr_utc"]
        - model_data["prediction_cutoff_utc"]
    ).dt.total_seconds().div(60).where(previous_leg_matches)
    model_data["minutes_since_previous_actual_arrival"] = (
        model_data["prediction_cutoff_utc"]
        - model_data["previous_actual_arr_utc"]
    ).dt.total_seconds().div(60).where(arrived_by_cutoff)

    features_created = [
        "tail_sequence_today", "has_previous_tail_flight",
        "previous_flight_matches_origin", "previous_arrived_by_cutoff",
        "previous_departed_by_cutoff", "previous_arrival_delay_known",
        "previous_departure_delay_known", "scheduled_turnaround_minutes",
        "minutes_until_previous_scheduled_arrival",
        "minutes_since_previous_actual_arrival",
    ]
    return model_data, features_created


### Recent operations

Recent operations summarize the state of SFO and carrier-specific activity before cutoff. 

Is the airport or this carrier backing up right now?

These are model inputs, not just summary statistics. They describe whether SFO, a carrier, or a carrier-destination pair has been backing up in the hours before prediction.


In [10]:
def add_recent_operation_features(
    flights_to_score,
    operations,
    group_columns,
    event_time,
    delay_column,
    prefix,
    windows_hours,
):
    """Summarize activity observed before each prediction cutoff."""
    model_data = flights_to_score.copy().reset_index(drop=True)
    prior_operations = operations.dropna(subset=[event_time]).copy()
    delays = pd.to_numeric(prior_operations[delay_column], errors="coerce")
    prior_operations["_count"] = 1.0
    prior_operations["_delay_valid"] = delays.notna().astype(float)
    prior_operations["_delay_total"] = delays.fillna(0)
    prior_operations["_delay_ge_15"] = delays.ge(15).astype(float)
    metrics = ["_count", "_delay_valid", "_delay_total", "_delay_ge_15"]
    features_created = []

    for hours in windows_hours:
        totals = calculate_window_totals(
            prior_operations,
            model_data,
            group_columns,
            event_time,
            metrics,
            pd.Timedelta(hours=int(hours)),
        )
        valid_delays = totals["_delay_valid"]
        new_columns = {
            f"{prefix}_observed_{hours}h": totals["_count"],
            f"{prefix}_delay_mean_{hours}h": np.divide(
                totals["_delay_total"],
                valid_delays,
                out=np.full(len(model_data), np.nan),
                where=valid_delays > 0,
            ),
            f"{prefix}_delay_ge_15_rate_{hours}h": np.divide(
                totals["_delay_ge_15"],
                valid_delays,
                out=np.full(len(model_data), np.nan),
                where=valid_delays > 0,
            ),
        }
        for name, values in new_columns.items():
            model_data[name] = values.astype("float32")
            features_created.append(name)

    return model_data, features_created


In [11]:
def add_operational_features(departures, all_flights):
    """Add recent SFO, carrier, and carrier-destination conditions."""
    sfo_departures = all_flights.loc[all_flights["origin"].eq(AIRPORT)]
    sfo_arrivals = all_flights.loc[
        all_flights["dest"].eq(AIRPORT)
        & all_flights["cancelled"].eq(0)
        & all_flights["diverted"].eq(0)
    ]
    airport_windows = [
        int(value)
        for value in FEATURES["recent_airport_operations_windows_hours"]
    ]
    route_windows = [
        int(value) for value in FEATURES["carrier_route_windows_hours"]
    ]
    carrier_windows = [
        int(value) for value in FEATURES["recent_carrier_operations_windows_hours"]
    ]

    model_data, airport_departure_features = add_recent_operation_features(
        departures,
        sfo_departures,
        (),
        "actual_dep_utc",
        "dep_delay_minutes",
        "sfo_departures",
        airport_windows,
    )
    model_data, airport_arrival_features = add_recent_operation_features(
        model_data,
        sfo_arrivals,
        (),
        "actual_arr_utc",
        "arr_delay_minutes",
        "sfo_arrivals",
        airport_windows,
    )
    model_data, carrier_departure_features = add_recent_operation_features(
        model_data,
        sfo_departures,
        ("reporting_airline",),
        "actual_dep_utc",
        "dep_delay_minutes",
        "carrier_sfo_departures",
        carrier_windows,
    )
    model_data, carrier_arrival_features = add_recent_operation_features(
        model_data,
        sfo_arrivals,
        ("reporting_airline",),
        "actual_arr_utc",
        "arr_delay_minutes",
        "carrier_sfo_arrivals",
        carrier_windows,
    )
    model_data, route_features = add_recent_operation_features(
        model_data,
        sfo_departures,
        ("reporting_airline", "dest"),
        "actual_dep_utc",
        "dep_delay_minutes",
        "carrier_destination_departures",
        route_windows,
    )

    return model_data, {
        "airport_traffic": [
            *airport_departure_features,
            *airport_arrival_features,
        ],
        "carrier_disruption": [
            *carrier_departure_features,
            *carrier_arrival_features,
            *route_features,
        ],
    }


### SFO weather

The latest SFO observation at or before cutoff supplies visibility, ceiling, wind, pressure, precipitation, and extreme-condition indicators.


In [12]:
def prepare_sfo_weather(weather):
    """Create weather levels and rolling extremes for SFO."""
    station_frames = []
    for station_id, station in weather.groupby("noaa_station_id", sort=False):
        station = station.sort_values("weather_observed_utc").drop_duplicates("weather_observed_utc", keep="last")
        indexed = station.set_index("weather_observed_utc")
        prior_weather = indexed.rolling(f"{WEATHER_PRIOR_WINDOW_HOURS}h", closed="right", min_periods=1)
        suffix = WEATHER_PRIOR_WINDOW_SUFFIX

        station[f"visibility_min_prior_{suffix}"] = prior_weather["visibility_km"].min().to_numpy()
        station[f"ceiling_min_prior_{suffix}"] = prior_weather["ceiling_height_m"].min().to_numpy()
        station[f"wind_speed_max_prior_{suffix}"] = prior_weather["wind_speed_mps"].max().to_numpy()
        station[f"wind_gust_max_prior_{suffix}"] = prior_weather["wind_gust_mps"].max().to_numpy()

        pressure = prior_weather["sea_level_pressure_hpa"]
        station[f"pressure_range_prior_{suffix}"] = (pressure.max() - pressure.min()).to_numpy()
        station["temperature_dewpoint_spread_c"] = station["temperature_c"] - station["dew_point_temperature_c"]

        wind_direction = pd.to_numeric(station["wind_direction_degrees"], errors="coerce")
        station["wind_direction_sin"] = np.sin(np.deg2rad(wind_direction))
        station["wind_direction_cos"] = np.cos(np.deg2rad(wind_direction))
        station["wind_gust_excess_mps"] = (station["wind_gust_mps"] - station["wind_speed_mps"]).clip(lower=0)

        known_conditions = station["visibility_km"].notna() | station["ceiling_height_m"].notna()
        flight_category = pd.Series(np.nan, index=station.index)
        flight_category.loc[known_conditions] = 0
        flight_category.loc[(station["visibility_km"].lt(8.047) | station["ceiling_height_m"].lt(914.4)) & known_conditions] = 1
        flight_category.loc[(station["visibility_km"].lt(4.828) | station["ceiling_height_m"].lt(304.8)) & known_conditions] = 2
        flight_category.loc[(station["visibility_km"].lt(1.609) | station["ceiling_height_m"].lt(152.4)) & known_conditions] = 3
        station["flight_category_code"] = flight_category

        station["is_near_freezing"] = station["temperature_c"].between(-2, 2).astype("int8")
        station["freezing_precipitation_proxy"] = (station["precipitation_mm"].gt(0) & station["temperature_c"].le(2)).astype("int8")
        station["is_low_visibility"] = station["visibility_km"].lt(5).astype("int8")
        station["is_low_ceiling"] = station["ceiling_height_m"].lt(1000).astype("int8")
        station["is_gusty"] = station["wind_gust_mps"].ge(10).astype("int8")
        station["noaa_station_id"] = station_id
        station_frames.append(station)

    return pd.concat(station_frames, ignore_index=True)

In [13]:
def add_weather_features(departures, weather):
    """Match each flight to the latest available SFO weather observation."""
    model_data = departures.copy().reset_index(drop=True)
    model_data["prediction_cutoff_utc"] = pd.to_datetime(model_data["prediction_cutoff_utc"], utc=True)
    model_data["_row_id"] = np.arange(len(model_data))

    sfo_weather = prepare_sfo_weather(weather)
    sfo_weather = sfo_weather[sfo_weather["noaa_station_id"].eq(NOAA_STATION_ID)].copy()
    sfo_weather["weather_observed_utc"] = pd.to_datetime(sfo_weather["weather_observed_utc"], utc=True)

    weather_columns = [
        column
        for column in sfo_weather.columns
        if column not in {"noaa_station_id", "weather_observed_utc"}
    ]

    flight_cutoffs = model_data[["_row_id", "prediction_cutoff_utc"]].sort_values("prediction_cutoff_utc")
    sfo_weather = sfo_weather.sort_values("weather_observed_utc")
    weather_tolerance = pd.Timedelta(hours=float(FEATURES["weather_tolerance_hours"]))

    weather_at_cutoff = pd.merge_asof(
        flight_cutoffs,
        sfo_weather,
        left_on="prediction_cutoff_utc",
        right_on="weather_observed_utc",
        direction="backward",
        tolerance=weather_tolerance,
    ).set_index("_row_id")

    observed_column = "origin_weather_observed_utc"
    model_data[observed_column] = model_data["_row_id"].map(weather_at_cutoff["weather_observed_utc"])

    features_created = []

    for column in weather_columns:
        feature = f"origin_{column}"
        model_data[feature] = model_data["_row_id"].map(weather_at_cutoff[column])
        features_created.append(feature)

    weather_age = model_data["prediction_cutoff_utc"] - model_data[observed_column]
    model_data["origin_weather_age_minutes"] = weather_age.dt.total_seconds() / 60
    features_created.append("origin_weather_age_minutes")

    model_data = model_data.drop(columns="_row_id")

    return model_data, features_created

### FAA aircraft context

FAA fields are joined for EDA by normalized tail number. They remain in the modeling table for analysis but are excluded from the final model features because they did not add stable predictive value beyond lineage.


In [14]:
def add_aircraft_features(departures, aircraft):
    """Join FAA aircraft details using the normalized tail number."""
    requested_columns = [
        "tail_number_key",
        "aircraft_year_manufactured",
        "aircraft_type_code",
        "engine_type_code",
        "aircraft_manufacturer",
        "aircraft_model",
        "number_of_engines",
        "number_of_seats",
        "aircraft_weight_class",
        "aircraft_cruise_speed",
        "engine_thrust",
        "faa_record_type",
    ]

    available_columns = [column for column in requested_columns if column in aircraft.columns]

    model_data = departures.merge(
        aircraft[available_columns],
        on="tail_number_key",
        how="left",
        validate="many_to_one",
    )

    manufactured_year = pd.to_numeric(model_data["aircraft_year_manufactured"], errors="coerce")
    flight_year = pd.to_datetime(model_data["flight_date"]).dt.year

    model_data["aircraft_age_years"] = flight_year - manufactured_year
    model_data["aircraft_age_years"] = model_data["aircraft_age_years"].where(
        model_data["aircraft_age_years"].between(0, 100)
    )

    features_created = [
        column for column in available_columns if column != "tail_number_key"
    ]
    features_created += ["aircraft_age_years"]

    return model_data, features_created

### Targets and delay bins - Outcomes!

For operated flights, `under_15` is on time; delay classes begin at 15, 60, 120, and 240 minutes. 

e.g. 

`target_delay_bin`:
- 0 = under_15
- 1 = 15_to_under_60
- 2 = 60_to_under_120
- 3 = 120_to_under_240
- 4 = 240_plus

In [15]:
def add_targets(model_data):
    """Create cancellation, binary delay, and five-bin outcomes."""
    model_data = model_data.copy()
    delay = pd.to_numeric(model_data["dep_delay_minutes"], errors="coerce")
    cancelled = model_data["cancelled"].eq(1)
    operated = ~cancelled & delay.notna()

    model_data["target_cancelled"] = cancelled.astype("int8")
    model_data["target_delay_ge_15"] = pd.Series(pd.NA, index=model_data.index, dtype="Int8")
    model_data.loc[operated, "target_delay_ge_15"] = delay.loc[operated].ge(15).astype("int8")

    delay_bins = pd.cut(
        delay,
        bins=[-np.inf, *DELAY_THRESHOLDS, np.inf],
        labels=False,
        right=False,
    )
    model_data["target_delay_bin"] = pd.Series(pd.NA, index=model_data.index, dtype="Int8")
    model_data.loc[operated, "target_delay_bin"] = delay_bins.loc[operated].astype("int8")
    label_lookup = dict(enumerate(BIN_LABELS))
    model_data["target_delay_bin_label"] = model_data["target_delay_bin"].map(label_lookup).astype("string")
    return model_data


### Build modeling table per horizon

Start with SFO departures, set the prediction cutoff, add each feature family in order, create targets, and write one parquet file per horizon. Same-tail lineage supports the tail-aware model. 

Build order: 
1. schedule
2. historical context
3. tail/recent/weather features
4. targets 


In [16]:
def build_horizon_dataset(horizon_hours):
    """Build the feature table for one prediction horizon."""
    output_path = PATHS.modeling / f"sfo_delay_features_{horizon_hours}h.parquet"

    if output_path.exists():
        model_data = pd.read_parquet(output_path)

    else:
        model_data = flights[
            flights["origin"].eq(AIRPORT)
            & flights["reporting_airline"].isin(TRACKED_AIRLINES)
        ].copy()

        model_data["prediction_cutoff_utc"] = model_data["scheduled_dep_utc"] - pd.Timedelta(hours=horizon_hours)

        # Remove flights that departed before the prediction cutoff
        departed_before_cutoff = (
            model_data["actual_dep_utc"].notna()
            & (model_data["actual_dep_utc"] <= model_data["prediction_cutoff_utc"])
        )
        model_data = model_data[~departed_before_cutoff].copy()

        feature_families = {}
        sfo_history = flights[flights["origin"].eq(AIRPORT)]
        lookback_days = FEATURES["historical_reliability_lookback_days"]

        model_data, feature_families["schedule"] = add_schedule_features(model_data, flights)
        model_data, feature_families["historical_reliability"] = add_historical_features(
            model_data, sfo_history, lookback_days
        )
        model_data, feature_families["historical_causes"] = add_cause_history(model_data, sfo_history)
        model_data, feature_families["lineage"] = add_lineage_features(model_data, flights)

        model_data, operation_features = add_operational_features(model_data, flights)
        feature_families.update(operation_features)

        model_data, feature_families["weather"] = add_weather_features(model_data, weather)
        model_data, feature_families["aircraft"] = add_aircraft_features(model_data, aircraft)

        model_data = add_targets(model_data)
        model_data["prediction_horizon_hours"] = horizon_hours

        # Keep delay causes for EDA, but exclude them from model features
        delay_cause_columns = [
            "carrier_delay_minutes",
            "weather_delay_minutes",
            "nas_delay_minutes",
            "security_delay_minutes",
            "late_aircraft_delay_minutes",
        ]

        for column in delay_cause_columns:
            model_data[f"eda_only_{column}"] = model_data[column]

        # Keep valid features and remove duplicate names
        cleaned_feature_families = {}

        for family, columns in feature_families.items():
            valid_columns = []

            for column in columns:
                if column in model_data.columns and column not in valid_columns:
                    valid_columns.append(column)

            cleaned_feature_families[family] = valid_columns

        feature_families = cleaned_feature_families

        selected_columns = (
            list(IDENTIFIER_COLUMNS)
            + list(TARGET_COLUMNS)
            + list(EDA_ONLY_COLUMNS)
        )

        for columns in feature_families.values():
            selected_columns.extend(columns)

        selected_columns = list(dict.fromkeys(selected_columns))

        model_data = model_data[selected_columns]
        model_data = model_data.sort_values("scheduled_dep_utc").reset_index(drop=True)

        PATHS.modeling.mkdir(parents=True, exist_ok=True)
        model_data.to_parquet(output_path, index=False)

    operated_count = model_data["target_delay_bin"].notna().sum()
    cancelled_count = model_data["target_cancelled"].sum()
    prior_arrival_coverage = model_data["previous_arrival_delay_known"].notna().mean() * 100
    weather_coverage = model_data["origin_weather_age_minutes"].notna().mean() * 100

    return {
        "horizon": f"T-{horizon_hours}",
        "rows": len(model_data),
        "operated_with_delay_label": int(operated_count),
        "cancelled": int(cancelled_count),
        "prior_arrival_known_percent": prior_arrival_coverage,
        "origin_weather_coverage_percent": weather_coverage,
    }

### Model table summary

Each horizon is built only if its cached parquet file is missing.

In [17]:
# Existing horizon files are reused; only missing files are rebuilt.
build_summaries = []

for horizon in HORIZONS:
    summary = build_horizon_dataset(horizon)
    build_summaries.append(summary)

feature_summary = pd.DataFrame(build_summaries).set_index("horizon")

display(feature_summary.round(2))

,rows,operated_with_delay_label,cancelled,prior_arrival_known_percent,origin_weather_coverage_percent
horizon,,,,,
T-2,454748,449387,5361,37.27,99.61
T-4,454748,449387,5361,28.56,99.62


### Generated Feature List

<details>
<summary><strong>Schedule and calendar features</strong></summary>

* `day_of_week`
* `scheduled_dep_hour`
* `scheduled_arr_hour`
* `month`
* `is_weekend`
* `hour_sin`
* `hour_cos`
* `month_sin`
* `month_cos`
* `days_until_holiday`
* `days_since_holiday`
* `scheduled_sfo_departures_same_hour`
* `scheduled_carrier_departures_same_hour`
* `distance_miles`
* `scheduled_elapsed_minutes`

</details>

<details>
<summary><strong>Historical reliability features</strong></summary>

Calculated separately for each:

* `carrier`
* `destination`
* `carrier_destination`

Using both `90d` and `365d` historical windows:

* `total_count`
* `operated_count`
* `cancel_rate`
* Delay rate for `>=15` minutes
* Delay rate for `>=60` minutes
* Delay rate for `>=120` minutes
* Delay rate for `>=240` minutes

</details>

<details>
<summary><strong>Historical delay-cause features</strong></summary>

Calculated separately for each:

* `carrier`
* `destination`
* `carrier_destination`

Using a `365d` historical window:

* `report_count`
* Carrier-delay positive rate
* NAS-delay positive rate
* Late-aircraft-delay positive rate
* Weather-delay positive rate

</details>

<details>
<summary><strong>Same-tail lineage features</strong></summary>

* `tail_sequence_today`
* `has_previous_tail_flight`
* `previous_flight_matches_origin`
* `previous_arrived_by_cutoff`
* `previous_departed_by_cutoff`
* `previous_arrival_delay_known`
* `previous_departure_delay_known`
* `scheduled_turnaround_minutes`
* `minutes_until_previous_scheduled_arrival`
* `minutes_since_previous_actual_arrival`

</details>

<details>
<summary><strong>Recent SFO operational features</strong></summary>

For SFO departures and arrivals over the prior `1h`, `3h`, and `6h`:

* `observed`
* `delay_mean`
* `delay_ge_15_rate`

</details>

<details>
<summary><strong>Recent carrier operations at SFO</strong></summary>

For carrier-specific SFO departures and arrivals over the prior `3h`, `6h`, and `24h`:

* `observed`
* `delay_mean`
* `delay_ge_15_rate`

</details>

<details>
<summary><strong>Recent carrier–destination operations</strong></summary>

For carrier–destination SFO departures over the prior `6h` and `24h`:

* `observed`
* `delay_mean`
* `delay_ge_15_rate`

</details>

<details>
<summary><strong>SFO weather features</strong></summary>

**Observed conditions**

* `origin_temperature_c`
* `origin_dew_point_temperature_c`
* `origin_relative_humidity_pct`
* `origin_visibility_km`
* `origin_wind_speed_mps`
* `origin_wind_gust_mps`
* `origin_wind_direction_degrees`
* `origin_sea_level_pressure_hpa`
* `origin_station_level_pressure_hpa`
* `origin_ceiling_height_m`
* `origin_altimeter_hpa`
* `origin_precipitation_mm`

**Prior configured-window conditions**

* `origin_visibility_min_prior_3h`
* `origin_ceiling_min_prior_3h`
* `origin_wind_speed_max_prior_3h`
* `origin_wind_gust_max_prior_3h`
* `origin_pressure_range_prior_3h`

**Derived weather features**

* `origin_temperature_dewpoint_spread_c`
* `origin_wind_direction_sin`
* `origin_wind_direction_cos`
* `origin_wind_gust_excess_mps`
* `origin_flight_category_code`
* `origin_is_near_freezing`
* `origin_freezing_precipitation_proxy`
* `origin_is_low_visibility`
* `origin_is_low_ceiling`
* `origin_is_gusty`
* `origin_weather_age_minutes`

</details>

<details>
<summary><strong>FAA aircraft context</strong></summary>

* `aircraft_year_manufactured`
* `aircraft_type_code`
* `engine_type_code`
* `aircraft_manufacturer`
* `aircraft_model`
* `number_of_engines`
* `number_of_seats`
* `aircraft_weight_class`
* `aircraft_cruise_speed`
* `engine_thrust`
* `faa_record_type`
* `aircraft_age_years`
* `faa_match_flag`

</details>

### Data leakage comments

- The model assumes the current tail number is known at the cutoff.
- A prior leg's schedule may be used once the tail is assigned, but its actual delay remains missing until the event is observed by cutoff.
- Recent operations use only events strictly before cutoff.
- SFO weather uses the latest observation at or before cutoff.
- Current flight causes, taxi times, and realized outcomes are not model features.
